# Step 8: Artifact Ablation -- Robustness Experiment

## 1. Objective

Steps 6 and 7 found that Random Forest and XGBoost both achieve near-perfect metrics, and that this is substantially explained by two engineered features (`amount_to_sender_balance`, `amount_exceeds_sender_balance`) that encode PaySim's synthetic "drained sender account" fraud-generation pattern. This notebook does **not** try to build a better model. It runs a controlled experiment: remove exactly those two features, retrain the same two model configurations unchanged, and measure how much performance actually changes. This is a robustness/ablation study, not model selection -- no new hyperparameters, no threshold tuning, no test-based model choice.

## 2. Why these two features are being removed

Step 2 (EDA) found that 97.50% of fraud TRANSFER/CASH_OUT transactions have `oldbalanceOrg == amount` and `newbalanceOrig == 0` (a fully-drained account), vs. **0%** of legitimate ones -- an almost perfectly separating pattern, flagged at the time as a likely PaySim simulation artifact rather than confirmed real fraud behavior. `amount_to_sender_balance` and `amount_exceeds_sender_balance` (Step 3) are the causal, leakage-safe features built to expose that relationship to a model. Steps 6/7 confirmed they dominate feature importance (45%+ combined in Random Forest, 41-53% combined gain in XGBoost). Removing them tests whether the models' strong scores depend on this one pattern, or whether the other 40 features (transaction, time, other balance, and behavioral/velocity features) carry meaningful signal on their own.

## 3. Feature-set verification

In [ ]:
import sys
sys.path.append("..")

import json
import os
import joblib
import numpy as np
import pandas as pd

from src.model_utils import (
    load_split_data,
    exclude_features,
    validate_model_features,
    build_random_forest,
    build_xgboost_classifier,
    train_model,
    tree_depth_summary,
    evaluate_classifier,
    evaluate_at_k,
    extract_tree_feature_importance,
    extract_xgboost_feature_importance,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

data = load_split_data()
feature_columns = data["feature_columns"]
artifact_features = ["amount_to_sender_balance", "amount_exceeds_sender_balance"]

print("Original feature count:", len(feature_columns))
assert len(feature_columns) == 42
for f in artifact_features:
    assert f in feature_columns
print("Both target artifact features confirmed present in the baseline 42-feature list.")

with open("../data/processed/split_metadata.json") as f:
    split_meta = json.load(f)
print("\nSplit ranges (unchanged from Step 4):", split_meta["train_step_range"], split_meta["validation_step_range"], split_meta["test_step_range"])
for name, y in [("train", data["y_train"]), ("validation", data["y_validation"]), ("test", data["y_test"])]:
    print(f"{name}: rows={len(y):,}, fraud={int(y.sum()):,}")

In [ ]:
ablated_columns = exclude_features(feature_columns, artifact_features)
print("Ablated feature count:", len(ablated_columns))
assert len(ablated_columns) == 40
for f in artifact_features:
    assert f not in ablated_columns

forbidden = ["nameOrig", "nameDest", "isFraud", "newbalanceOrig", "newbalanceDest", "isFlaggedFraud"]
for f in forbidden:
    assert f not in ablated_columns
print("Confirmed: 40 features, both artifact features absent, no ID/target/post-transaction columns present.")

X_train = data["X_train"][ablated_columns]
y_train = data["y_train"]
X_val = data["X_validation"][ablated_columns]
y_val = data["y_validation"]
X_test = data["X_test"][ablated_columns]
y_test = data["y_test"]

for split_name, X in [("train", X_train), ("validation", X_val), ("test", X_test)]:
    checks = validate_model_features(X, ablated_columns, expected_count=40)
    assert all(checks.values()), f"{split_name} failed: {checks}"
    print(f"{split_name}: {checks}")

## 4. Model configuration

Identical to Step 6 (Model G) and Step 7's unweighted variant (Model H) -- only the feature matrix changes, nothing else. No `scale_pos_weight`, no tuning.

In [ ]:
rf_config = dict(n_estimators=200, max_depth=20, min_samples_split=2, min_samples_leaf=1,
                  max_features="sqrt", n_jobs=-1, random_state=42)
xgb_config = dict(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                   min_child_weight=1, reg_lambda=1.0, tree_method="hist", random_state=42, n_jobs=8)
print("Model G (Random Forest, unweighted) config:", rf_config)
print("Model H (XGBoost, unweighted, scale_pos_weight=1) config:", xgb_config)

## 5. Training

In [ ]:
rf_g = build_random_forest(class_weight=None, **rf_config)
info_g = train_model(rf_g, X_train, y_train)
depth_g = tree_depth_summary(info_g["pipeline"])
print(f"Model G: training_time_sec={info_g['training_time_sec']:.1f}, tree depth={depth_g}")

In [ ]:
model_h = build_xgboost_classifier(scale_pos_weight=1.0, **xgb_config)
info_h = train_model(model_h, X_train, y_train)
print(f"Model H: training_time_sec={info_h['training_time_sec']:.1f}, boosting rounds={info_h['pipeline'].get_booster().num_boosted_rounds()}")

fitted = {"model_g_rf_ablated": info_g, "model_h_xgb_ablated": info_h}
print("\nBoth fit ONLY on (X_train, y_train) -- the ablated 40-column matrix.")

**Observation:** Model G's trees now hit `max_depth=20` uniformly (min depth = mean depth = max depth = 20.0, vs. Step 6's min 16 / mean 19.835) and training took noticeably longer (see section 11) -- a hint that, without the two dominant features, the forest needs its full depth budget to find comparable structure.

## 6. Validation evaluation (threshold = 0.5)

In [ ]:
results = {}
for name, info in fitted.items():
    proba_val = info["pipeline"].predict_proba(X_val)[:, 1]
    info["proba_val"] = proba_val
    val_metrics = evaluate_classifier(y_val, proba_val, threshold=0.5)
    results.setdefault(name, {})["validation_metrics"] = val_metrics
    print(f"{name} -- VALIDATION @0.5: {val_metrics}\n")

## 7. Test evaluation -- FINAL TEST RESULTS

In [ ]:
for name, info in fitted.items():
    proba_test = info["pipeline"].predict_proba(X_test)[:, 1]
    info["proba_test"] = proba_test
    test_metrics = evaluate_classifier(y_test, proba_test, threshold=0.5)
    results[name]["test_metrics"] = test_metrics
    print(f"{name} -- FINAL TEST RESULTS @0.5: {test_metrics}\n")

## 8. Precision@K / Recall@K

In [ ]:
k_values = [100, 500, 1000, 5000, 10000]
for name, info in fitted.items():
    at_k_val = evaluate_at_k(y_val, info["proba_val"], k_values)
    at_k_test = evaluate_at_k(y_test, info["proba_test"], k_values)
    results[name]["at_k_validation"] = at_k_val.to_dict(orient="records")
    results[name]["at_k_test"] = at_k_test.to_dict(orient="records")
    print(f"{name} -- VALIDATION:\n{at_k_val.to_string(index=False)}")
    print(f"\n{name} -- TEST:\n{at_k_test.to_string(index=False)}\n")

## 9. Feature importance

Same methodology as Steps 6/7 (impurity-based for Random Forest, gain-based for XGBoost). The two removed features cannot appear -- they were never in the feature matrix -- confirmed explicitly below.

In [ ]:
imp_rf = extract_tree_feature_importance(fitted["model_g_rf_ablated"]["pipeline"], ablated_columns)
imp_xgb = extract_xgboost_feature_importance(fitted["model_h_xgb_ablated"]["pipeline"], ablated_columns, importance_type="gain")

print("=== Model G (Random Forest, ablated) Top 20 ===")
print(imp_rf.head(20).to_string(index=False))
print(f"\nArtifact features absent from RF importance table: {not any(f in imp_rf['feature'].values for f in artifact_features)}")

print("\n=== Model H (XGBoost, ablated) Top 20 ===")
print(imp_xgb.head(20).to_string(index=False))
print(f"\nArtifact features absent from XGB importance table: {not any(f in imp_xgb['feature'].values for f in artifact_features)}")

**Finding:** with the two artifact features removed, importance redistributes to `oldbalanceOrg`, `amount`/`log_amount`, and `hour_of_day` for Random Forest, and to `destination_balance_zero`, `is_transfer_or_cash_out`, and `prior_receiver_has_history` for XGBoost. These are genuine, still-causal, still-available-at-decision-time features -- so the models are not left with nothing; they retain real signal, just weaker than the removed pattern.

## 10. Comparison with Step 6/7

In [ ]:
with open("../results/random_forest_metrics.json") as f:
    rf_step6 = json.load(f)
with open("../results/xgboost_metrics.json") as f:
    xgb_step7 = json.load(f)

rf_baseline = rf_step6["model_c_unweighted"]
xgb_baseline = xgb_step7["model_e_unweighted"]
rf_ablated = results["model_g_rf_ablated"]
xgb_ablated = results["model_h_xgb_ablated"]

def diff_row(model_pair_name, baseline, ablated, split):
    b, a = baseline[f"{split}_metrics"], ablated[f"{split}_metrics"]
    return {
        "comparison": model_pair_name, "split": split,
        "precision_before": b["precision"], "precision_after": a["precision"], "precision_change": a["precision"] - b["precision"],
        "recall_before": b["recall"], "recall_after": a["recall"], "recall_change": a["recall"] - b["recall"],
        "f1_before": b["f1"], "f1_after": a["f1"], "f1_change": a["f1"] - b["f1"],
        "roc_auc_before": b["roc_auc"], "roc_auc_after": a["roc_auc"], "roc_auc_change": a["roc_auc"] - b["roc_auc"],
        "pr_auc_before": b["pr_auc"], "pr_auc_after": a["pr_auc"], "pr_auc_change": a["pr_auc"] - b["pr_auc"],
    }

rows = []
for split in ["validation", "test"]:
    rows.append(diff_row("Random Forest: Step 6 (42 feat) vs Step 8 ablated (40 feat)", rf_baseline, rf_ablated, split))
    rows.append(diff_row("XGBoost: Step 7 (42 feat) vs Step 8 ablated (40 feat)", xgb_baseline, xgb_ablated, split))

comparison_df = pd.DataFrame(rows)
comparison_df

This is reported as a measured before/after difference, not a ranking or a "winner" -- the point is the size of the gap, not which model is better.

## 11. Interpretation

**Yes, performance drops substantially after removing the two features, for both models, on both validation and test.**

- **Random Forest** shows the larger drop: validation PR-AUC falls from 0.9998 to 0.7265 (-0.2733), recall at threshold 0.5 falls from 0.9947 to 0.5408 (-0.4539). On test, PR-AUC falls from 0.9997 to 0.7396 (-0.2601) and recall from 0.9940 to 0.6146 (-0.3794). ROC-AUC barely moves (validation -0.0058, test -0.0096) -- a reminder that ROC-AUC is far less sensitive to this kind of change than PR-AUC/recall are, exactly the point made in Steps 5-7 about not relying on ROC-AUC alone for a rare-event problem.
- **XGBoost** also drops, but by less: validation PR-AUC falls from 0.9817 to 0.8390 (-0.1427), recall from 0.8812 to 0.6915 (-0.1897). Test PR-AUC falls from 0.9971 to 0.9191 (-0.0779), recall from 0.9019 to 0.7526 (-0.1493).

**This confirms Step 7's finding was directionally correct: a meaningful share of both models' strong Step 6/7 performance was attributable to the two artifact features.** But the drop is not to a random-guessing floor -- both ablated models still retain real, substantial predictive power (Model H's test PR-AUC of 0.9191 and recall of 0.7526 are still far better than either Logistic Regression baseline from Step 5). This means: **other transaction, balance, and behavioral features do carry genuine signal on their own**, just less separable than the removed pattern. XGBoost's smaller relative drop (vs. Random Forest's larger one) suggests its sequential boosting was better able to compensate using the remaining 40 features, though this experiment does not attempt to explain that mechanistically beyond what feature importance in section 9 already shows.

**On generalization:** none of this establishes how these models would perform on real transaction data. The two removed features encoded a documented PaySim simulation artifact (Steps 2/3); their large measured effect here is itself evidence that a meaningful part of Steps 6-7's headline numbers reflected fitting to that specific synthetic pattern, not a demonstrated real-world fraud-detection capability. **PaySim is a synthetic dataset, so these results should be interpreted as an experimental benchmark rather than evidence of production fraud-detection performance.** This model is not claimed to be production-ready, and no claim is made about performance on real UPI/NPCI or other real-world payment transaction data.

## 12. Limitations

- This is a single ablation (removing exactly two features together); it does not isolate which of the two contributes more, nor test other feature subsets -- that would need a separate, further-controlled experiment.
- No hyperparameter re-tuning was performed for the ablated feature set -- the Step 6/7 configurations may not be optimal for 40 features, so some of the measured drop could reflect a configuration mismatch rather than pure information loss. This was a deliberate choice (to isolate the effect of the features alone) but is a real limitation of the comparison.
- Test-period distribution shift (Steps 2/4) still applies -- test fraud rate (0.4361%) differs substantially from train (0.0816%), so absolute test metrics here are not directly comparable to a real deployment's base rate.
- This remains a synthetic-dataset experiment; conclusions are scoped to PaySim and are not extrapolated to real payment systems.

## Save artifacts

In [ ]:
joblib.dump(fitted["model_g_rf_ablated"]["pipeline"], "../models/random_forest_ablation_no_balance_ratio.joblib")
joblib.dump(fitted["model_h_xgb_ablated"]["pipeline"], "../models/xgboost_ablation_no_balance_ratio.joblib")
print("Saved models/random_forest_ablation_no_balance_ratio.joblib and models/xgboost_ablation_no_balance_ratio.joblib")

for name, path in [
    ("model_g_rf_ablated", "../models/random_forest_ablation_no_balance_ratio.joblib"),
    ("model_h_xgb_ablated", "../models/xgboost_ablation_no_balance_ratio.joblib"),
]:
    reloaded = joblib.load(path)
    reloaded_proba = reloaded.predict_proba(X_val)[:, 1]
    max_diff = np.max(np.abs(fitted[name]["proba_val"] - reloaded_proba))
    same_class = np.array_equal((fitted[name]["proba_val"] >= 0.5).astype(int), (reloaded_proba >= 0.5).astype(int))
    print(f"{name}: max_abs_diff={max_diff:.2e}, classifications identical @0.5={same_class}")

print("\nStep 5-7 artifacts untouched:")
for f in ["../models/logistic_regression_baseline.joblib", "../models/logistic_regression_balanced.joblib",
          "../models/random_forest_baseline.joblib", "../models/random_forest_balanced.joblib",
          "../models/xgboost_baseline.joblib", "../models/xgboost_balanced.joblib"]:
    print(f"  {f}: exists = {os.path.exists(f)}")

In [ ]:
metrics_rows = []
for name, r in results.items():
    for split in ["validation", "test"]:
        m = r[f"{split}_metrics"]
        metrics_rows.append({
            "model": name, "split": split, "precision": m["precision"], "recall": m["recall"], "f1": m["f1"],
            "roc_auc": m["roc_auc"], "pr_auc": m["pr_auc"],
            "tn": m["confusion_matrix"]["tn"], "fp": m["confusion_matrix"]["fp"],
            "fn": m["confusion_matrix"]["fn"], "tp": m["confusion_matrix"]["tp"],
        })
pd.DataFrame(metrics_rows).to_csv("../results/artifact_ablation_metrics.csv", index=False)

at_k_rows = []
for name, r in results.items():
    for split_name, records in [("validation", r["at_k_validation"]), ("test", r["at_k_test"])]:
        for rec in records:
            at_k_rows.append({"model": name, "split": split_name, **rec})
pd.DataFrame(at_k_rows).to_csv("../results/artifact_ablation_precision_recall_at_k.csv", index=False)

imp_rf.to_csv("../results/artifact_ablation_feature_importance_rf.csv", index=False)
imp_xgb.to_csv("../results/artifact_ablation_feature_importance_xgb.csv", index=False)
comparison_df.to_csv("../results/artifact_ablation_comparison.csv", index=False)

print("Saved results/artifact_ablation_metrics.csv")
print("Saved results/artifact_ablation_precision_recall_at_k.csv")
print("Saved results/artifact_ablation_feature_importance_rf.csv")
print("Saved results/artifact_ablation_feature_importance_xgb.csv")
print("Saved results/artifact_ablation_comparison.csv")